# Exercícios — SQL Intermediário com SQLite — Resolução

Este notebook traz **12 exercícios** com um único tema: **uma livraria com autores, livros, pedidos e itens de pedido**.

Os exercícios 1 e 2 trabalham a estrutura das tabelas, PK, FK, PK composta e DEFAULT. A partir do exercício 3, vamos direto para INNER JOIN, LEFT JOIN, relações 1:N e N:N e agregações com várias tabelas.

Cada exercício tem um **enunciado** e a **resolução comentada**. Tente resolver no notebook do aluno antes de consultar as respostas!

Rode a preparação abaixo primeiro. Vamos usar uma única `conexao` e um único `cursor`.

Baixe também `livraria_aula2.db` e coloque o arquivo na pasta em que você vai executar o notebook. No Colab, envie o banco pela aba **Arquivos**. SQLite já faz parte do Python.


In [1]:
import sqlite3
from pathlib import Path

# Se o arquivo estiver em outra pasta, ajuste este caminho.
caminho_banco = Path("livraria_aula2.db")

# Localiza o banco também ao executar este material dentro do repositório.
if not caminho_banco.is_file():
    caminhos = [
        Path("Aula 2 - Intermediário/livraria_aula2.db"),
        Path("../livraria_aula2.db"),
        Path("../../livraria_aula2.db"),
    ]
    caminho_banco = next((p for p in caminhos if p.is_file()), caminho_banco)

assert caminho_banco.is_file(), "Baixe livraria_aula2.db e ajuste caminho_banco."

conexao = sqlite3.connect(caminho_banco)
cursor = conexao.cursor()
cursor.execute("PRAGMA foreign_keys = ON")
print("Banco conectado:", caminho_banco)


Banco conectado: ../../livraria_aula2.db


## O banco dos exercícios: uma livraria

| Tabela | Colunas | Chaves |
| --- | --- | --- |
| autores | id_autor (INTEGER), nome (TEXT), pais (TEXT) | PK: id_autor |
| livros | id_livro (INTEGER), id_autor (INTEGER), titulo (TEXT), genero (TEXT), preco (REAL) | PK: id_livro; FK: id_autor → autores |
| pedidos | id_pedido (INTEGER), data_pedido (TEXT) | PK: id_pedido |
| itens_pedido | id_pedido (INTEGER), id_livro (INTEGER), quantidade (INTEGER, DEFAULT 1) | PK: (id_pedido, id_livro); duas FKs |

**Autor → Livro é 1:N:** um autor pode ter vários livros; nesta base, cada livro tem um único autor.

**Livro ↔ Pedido é N:N:** um pedido pode conter vários livros e um livro pode aparecer em vários pedidos. A tabela `itens_pedido` faz a ligação.

O banco tem **20 autores, 40 livros, 250 pedidos e 853 itens**. Autores e títulos são fictícios. Há 3 autores sem livro e 5 livros nunca vendidos. As datas usam TEXT no formato `AAAA-MM-DD`.

Neste modelo, o preço do catálogo é usado para calcular o valor das vendas.

Para executar as consultas, use `cursor.execute(""" ... """)` e depois `cursor.fetchall()`. No Python, NULL aparece como `None`.


## Exercício 1 — Conhecendo a estrutura da livraria

Antes de escrever JOIN, vamos olhar as tabelas do banco fornecido.

**a)** Consulte a estrutura de `autores`, `livros`, `pedidos` e `itens_pedido` com `PRAGMA table_info`.

**b)** Consulte as FKs de `livros` e `itens_pedido` com `PRAGMA foreign_key_list`.

Depois de rodar, responda: quais colunas são INTEGER, TEXT e REAL? Qual tabela tem PK composta? Para quais tabelas as FKs apontam? Em qual coluna o valor padrão é 1?

> `pk = 1` e `pk = 2` são posições na mesma PK. `dflt_value` mostra o padrão declarado em DEFAULT.


In [2]:
# a) Uma linha por coluna, com tipo, obrigatoriedade, padrão e posição na PK.
for tabela in ['autores', 'livros', 'pedidos', 'itens_pedido']:
    cursor.execute(f"PRAGMA table_info({tabela})")
    print(tabela, cursor.fetchall())


autores [(0, 'id_autor', 'INTEGER', 0, None, 1), (1, 'nome', 'TEXT', 1, None, 0), (2, 'pais', 'TEXT', 1, None, 0)]
livros [(0, 'id_livro', 'INTEGER', 0, None, 1), (1, 'id_autor', 'INTEGER', 1, None, 0), (2, 'titulo', 'TEXT', 1, None, 0), (3, 'genero', 'TEXT', 1, None, 0), (4, 'preco', 'REAL', 1, None, 0)]
pedidos [(0, 'id_pedido', 'INTEGER', 0, None, 1), (1, 'data_pedido', 'TEXT', 1, None, 0)]
itens_pedido [(0, 'id_pedido', 'INTEGER', 1, None, 1), (1, 'id_livro', 'INTEGER', 1, None, 2), (2, 'quantidade', 'INTEGER', 1, '1', 0)]


In [3]:
# b) "from" é a coluna local; "table" e "to" mostram a referência.
for tabela in ['livros', 'itens_pedido']:
    cursor.execute(f"PRAGMA foreign_key_list({tabela})")
    print(tabela, cursor.fetchall())

# IDs e quantidade: INTEGER; nomes, títulos, país, gênero e data: TEXT; preco: REAL.
# livros.id_autor aponta para autores.id_autor.
# itens_pedido tem PK (id_pedido, id_livro), com FKs para pedidos e livros.
# itens_pedido.quantidade tem DEFAULT 1.


livros [(0, 0, 'autores', 'id_autor', 'id_autor', 'NO ACTION', 'NO ACTION', 'NONE')]
itens_pedido [(0, 0, 'livros', 'id_livro', 'id_livro', 'NO ACTION', 'NO ACTION', 'NONE'), (1, 0, 'pedidos', 'id_pedido', 'id_pedido', 'NO ACTION', 'NO ACTION', 'NONE')]


## Exercício 2 — Criando as tabelas da livraria

Use o mesmo tema da aula: **autores, livros, pedidos e itens de pedido**.

**a)** Crie `autores_treino` com `id_autor INTEGER PRIMARY KEY`, `nome TEXT NOT NULL` e `pais TEXT NOT NULL`.

**b)** Crie `livros_treino` com `id_livro INTEGER PRIMARY KEY`, `id_autor INTEGER NOT NULL`, `titulo TEXT NOT NULL`, `genero TEXT NOT NULL` e `preco REAL NOT NULL`. Declare a FK para `autores_treino`.

**c)** Crie `pedidos_treino` com `id_pedido INTEGER PRIMARY KEY` e `data_pedido TEXT NOT NULL`.

**d)** Crie `itens_pedido_treino` com duas FKs (`id_pedido` e `id_livro`), uma PK composta por essas duas colunas e `quantidade INTEGER NOT NULL DEFAULT 1`.

**e)** Insira 3 autores, 3 livros, 2 pedidos e 4 itens. Um autor deve ter mais de um livro. Um livro deve aparecer em mais de um pedido. Omita quantidade em uma inserção e informe quantidade 2 em outra.

**f)** Consulte a estrutura e os dados de `itens_pedido_treino`. Confira o valor padrão. Explique o que aconteceria ao repetir um par da PK ou informar uma FK inexistente.

**g)** Remova as tabelas de treino, começando pelas que possuem FKs.

> Use `CREATE TEMP TABLE`: as tabelas de treino existem apenas nesta conexão. Trabalhe com o mesmo `cursor` da preparação. Nas consultas seguintes, use as tabelas do banco fornecido, sem o sufixo `_treino`.


In [4]:
# a–d) Primeiro as tabelas referenciadas, depois as que possuem FKs.
cursor.execute("""
CREATE TEMP TABLE autores_treino (
    id_autor INTEGER PRIMARY KEY,
    nome TEXT NOT NULL,
    pais TEXT NOT NULL
)
""")
cursor.execute("""
CREATE TEMP TABLE livros_treino (
    id_livro INTEGER PRIMARY KEY,
    id_autor INTEGER NOT NULL,
    titulo TEXT NOT NULL,
    genero TEXT NOT NULL,
    preco REAL NOT NULL,
    FOREIGN KEY (id_autor) REFERENCES autores_treino(id_autor)
)
""")
cursor.execute("""
CREATE TEMP TABLE pedidos_treino (
    id_pedido INTEGER PRIMARY KEY,
    data_pedido TEXT NOT NULL
)
""")
cursor.execute("""
CREATE TEMP TABLE itens_pedido_treino (
    id_pedido INTEGER NOT NULL,
    id_livro INTEGER NOT NULL,
    quantidade INTEGER NOT NULL DEFAULT 1,
    PRIMARY KEY (id_pedido, id_livro),
    FOREIGN KEY (id_pedido) REFERENCES pedidos_treino(id_pedido),
    FOREIGN KEY (id_livro) REFERENCES livros_treino(id_livro)
)
""")
print('Quatro tabelas de treino criadas.')


Quatro tabelas de treino criadas.


In [5]:
# e) Nomes e títulos fictícios; um autor pode ter vários livros.
cursor.execute("""
INSERT INTO autores_treino VALUES
    (1, 'Ana Ribeiro', 'Brasil'),
    (2, 'Bruno Santos', 'Portugal'),
    (3, 'Carla Mendes', 'Angola')
""")
cursor.execute("""
INSERT INTO livros_treino VALUES
    (1, 1, 'A cidade de papel', 'Romance', 30.0),
    (2, 1, 'O mapa das estrelas', 'Fantasia', 45.0),
    (3, 2, 'Cartas de inverno', 'Poesia', 25.0)
""")
cursor.execute("""
INSERT INTO pedidos_treino VALUES
    (1, '2026-01-15'), (2, '2026-01-16')
""")
# Quantidade omitida: DEFAULT 1.
cursor.execute("""
INSERT INTO itens_pedido_treino (id_pedido, id_livro) VALUES
    (1, 1), (2, 1)
""")
# Quantidade informada: o banco guarda 2.
cursor.execute("""
INSERT INTO itens_pedido_treino VALUES (1, 2, 2), (2, 3, 2)
""")
conexao.commit()


In [6]:
# f) Quantidade tem dflt_value = '1'; os IDs têm pk = 1 e pk = 2.
cursor.execute("PRAGMA table_info(itens_pedido_treino)")
print(cursor.fetchall())
cursor.execute("PRAGMA foreign_key_list(itens_pedido_treino)")
print(cursor.fetchall())
cursor.execute("SELECT * FROM itens_pedido_treino ORDER BY id_pedido, id_livro")
print(cursor.fetchall())

# Repetir o par viola a PK. Uma referência inexistente viola a FK.
# DEFAULT é usado quando a coluna é omitida; NULL explícito viola NOT NULL.


[(0, 'id_pedido', 'INTEGER', 1, None, 1), (1, 'id_livro', 'INTEGER', 1, None, 2), (2, 'quantidade', 'INTEGER', 1, '1', 0)]
[(0, 0, 'livros_treino', 'id_livro', 'id_livro', 'NO ACTION', 'NO ACTION', 'NONE'), (1, 0, 'pedidos_treino', 'id_pedido', 'id_pedido', 'NO ACTION', 'NO ACTION', 'NONE')]
[(1, 1, 1), (1, 2, 2), (2, 1, 1), (2, 3, 2)]


In [7]:
# g) Os itens dependem de pedidos e livros; os livros dependem de autores.
cursor.execute("DROP TABLE itens_pedido_treino")
cursor.execute("DROP TABLE pedidos_treino")
cursor.execute("DROP TABLE livros_treino")
cursor.execute("DROP TABLE autores_treino")
conexao.commit()


## Exercício 3 — INNER JOIN: quem escreveu cada livro?

Junte `autores` e `livros` para mostrar `id_livro`, `titulo`, o nome do autor com o nome `autor`, `pais` e `preco`.

Mostre os **10 primeiros livros**, em ordem de `id_livro`.

> Dica: a ligação é `autores.id_autor = livros.id_autor`.


In [8]:
cursor.execute("""
SELECT livros.id_livro, livros.titulo, autores.nome AS autor,
       autores.pais, livros.preco
FROM autores
INNER JOIN livros ON autores.id_autor = livros.id_autor
ORDER BY livros.id_livro
LIMIT 10
""")
cursor.fetchall()


[(1, 'A cidade de papel', 'Ana Ribeiro', 'Brasil', 104.93), (2, 'O mapa das estrelas', 'Bruno Santos', 'Portugal', 28.13), (3, 'Cartas de inverno', 'Carla Mendes', 'Angola', 59.38), (4, 'A última estação', 'Diego Costa', 'Moçambique', 52.9), (5, 'O jardim secreto do tempo', 'Elisa Rocha', 'Cabo Verde', 117.06), (6, 'Caminhos de areia', 'Felipe Lima', 'Brasil', 109.59), (7, 'O farol azul', 'Gabriela Alves', 'Portugal', 136.52), (8, 'Memórias de um amanhã', 'Henrique Dias', 'Angola', 35.87), (9, 'A ponte entre mundos', 'Isabela Duarte', 'Moçambique', 77.74), (10, 'O silêncio da praça', 'João Martins', 'Cabo Verde', 28.72)]

## Exercício 4 — LEFT JOIN: todos os autores

Liste **todos os autores**, inclusive os que não têm nenhum livro cadastrado.

Mostre `id_autor`, `nome` e `id_livro`. Ordene por `id_autor` e depois por `id_livro`.

Depois de rodar, responda: por que um autor pode aparecer em mais de uma linha? O que aparece em `id_livro` para quem não tem livro?


In [9]:
# Cada livro gera uma linha; autores sem livro aparecem com NULL (None no Python).
cursor.execute("""
SELECT autores.id_autor, autores.nome, livros.id_livro
FROM autores
LEFT JOIN livros ON autores.id_autor = livros.id_autor
ORDER BY autores.id_autor, livros.id_livro
""")
cursor.fetchall()


[(1, 'Ana Ribeiro', 1), (1, 'Ana Ribeiro', 18), (1, 'Ana Ribeiro', 35), (2, 'Bruno Santos', 2), (2, 'Bruno Santos', 19), (2, 'Bruno Santos', 36), (3, 'Carla Mendes', 3), (3, 'Carla Mendes', 20), (3, 'Carla Mendes', 37), (4, 'Diego Costa', 4), (4, 'Diego Costa', 21), (4, 'Diego Costa', 38), (5, 'Elisa Rocha', 5), (5, 'Elisa Rocha', 22), (5, 'Elisa Rocha', 39), (6, 'Felipe Lima', 6), (6, 'Felipe Lima', 23), (6, 'Felipe Lima', 40), (7, 'Gabriela Alves', 7), (7, 'Gabriela Alves', 24), (8, 'Henrique Dias', 8), (8, 'Henrique Dias', 25), (9, 'Isabela Duarte', 9), (9, 'Isabela Duarte', 26), (10, 'João Martins', 10), (10, 'João Martins', 27), (11, 'Larissa Nunes', 11), (11, 'Larissa Nunes', 28), (12, 'Marcos Vieira', 12), (12, 'Marcos Vieira', 29), (13, 'Natalia Lopes', 13), (13, 'Natalia Lopes', 30), (14, 'Otavio Souza', 14), (14, 'Otavio Souza', 31), (15, 'Paula Fernandes', 15), (15, 'Paula Fernandes', 32), (16, 'Rafael Gomes', 16), (16, 'Rafael Gomes', 33), (17, 'Sabrina Melo', 17), (17, 'Sa

## Exercício 5 — Autores sem livro

Encontre os autores que **não têm nenhum livro cadastrado**.

Mostre `id_autor`, `nome` e `pais`, em ordem de `id_autor`. Quantos autores estão nessa situação?

> Dica: depois do LEFT JOIN, procure `livros.id_livro IS NULL`.


In [10]:
# O resultado tem 3 autores sem livro cadastrado.
cursor.execute("""
SELECT autores.id_autor, autores.nome, autores.pais
FROM autores
LEFT JOIN livros ON autores.id_autor = livros.id_autor
WHERE livros.id_livro IS NULL
ORDER BY autores.id_autor
""")
cursor.fetchall()


[(18, 'Tiago Barros', 'Angola'), (19, 'Vanessa Pinto', 'Moçambique'), (20, 'Wesley Castro', 'Cabo Verde')]

## Exercício 6 — Livros nunca vendidos

Encontre os livros que **não aparecem em nenhum item de pedido**.

Mostre `id_livro`, `titulo` e `genero`, em ordem de `id_livro`. Quantos livros nunca foram vendidos?

> Dica: a relação agora é entre `livros` e `itens_pedido`.


In [11]:
# O resultado tem 5 livros sem venda.
cursor.execute("""
SELECT livros.id_livro, livros.titulo, livros.genero
FROM livros
LEFT JOIN itens_pedido ON livros.id_livro = itens_pedido.id_livro
WHERE itens_pedido.id_livro IS NULL
ORDER BY livros.id_livro
""")
cursor.fetchall()


[(36, 'O retrato perdido', 'Romance'), (37, 'A noite dos vaga-lumes', 'Fantasia'), (38, 'O rio que voltava', 'Suspense'), (39, 'Fragmentos de sol', 'Poesia'), (40, 'A estação das descobertas', 'Ficção científica')]

## Exercício 7 — JOIN em cadeia e valor de cada item

Junte as **quatro tabelas** e mostre `id_pedido`, `data_pedido`, `autor`, `livro`, `quantidade`, `preco` e o valor do item com o nome `subtotal`.

O subtotal é `quantidade * preco`, arredondado para 2 casas decimais.

Mostre as **15 primeiras linhas**, ordenadas por `id_pedido` e `id_livro`.

> Dica: siga o caminho autor → livro → item → pedido.


In [12]:
cursor.execute("""
SELECT pedidos.id_pedido, pedidos.data_pedido, autores.nome AS autor,
       livros.titulo AS livro, itens_pedido.quantidade, livros.preco,
       ROUND(itens_pedido.quantidade * livros.preco, 2) AS subtotal
FROM autores
JOIN livros ON autores.id_autor = livros.id_autor
JOIN itens_pedido ON livros.id_livro = itens_pedido.id_livro
JOIN pedidos ON itens_pedido.id_pedido = pedidos.id_pedido
ORDER BY pedidos.id_pedido, livros.id_livro
LIMIT 15
""")
cursor.fetchall()


[(1, '2026-02-05', 'Ana Ribeiro', 'A cidade de papel', 3, 104.93, 314.79), (1, '2026-02-05', 'Otavio Souza', 'Pequenos universos', 3, 49.85, 149.55), (1, '2026-02-05', 'Bruno Santos', 'A biblioteca das marés', 2, 126.18, 252.36), (2, '2026-01-24', 'Bruno Santos', 'O mapa das estrelas', 1, 28.13, 28.13), (2, '2026-01-24', 'Diego Costa', 'A última estação', 1, 52.9, 52.9), (2, '2026-01-24', 'Felipe Lima', 'Caminhos de areia', 3, 109.59, 328.77), (2, '2026-01-24', 'Ana Ribeiro', 'O espelho da lua', 3, 98.66, 295.98), (2, '2026-01-24', 'Larissa Nunes', 'O céu de outro planeta', 2, 37.09, 74.18), (3, '2026-04-27', 'Ana Ribeiro', 'A cidade de papel', 2, 104.93, 209.86), (3, '2026-04-27', 'Carla Mendes', 'Cartas de inverno', 1, 59.38, 59.38), (3, '2026-04-27', 'Larissa Nunes', 'O céu de outro planeta', 1, 37.09, 37.09), (4, '2026-05-29', 'Diego Costa', 'A última estação', 4, 52.9, 211.6), (4, '2026-05-29', 'João Martins', 'O silêncio da praça', 2, 28.72, 57.44), (5, '2026-02-10', 'Elisa Rocha

## Exercício 8 — Valor total de cada pedido

Calcule o valor total de cada pedido somando os valores de seus itens.

Mostre `id_pedido` e `total_pedido`, arredondado para 2 casas decimais. Liste os **10 pedidos de maior valor**, desempatando pelo `id_pedido`.

> Dica: use JOIN, GROUP BY e SUM(quantidade * preco).


In [13]:
cursor.execute("""
SELECT pedidos.id_pedido,
       ROUND(SUM(itens_pedido.quantidade * livros.preco), 2) AS total_pedido
FROM pedidos
JOIN itens_pedido ON pedidos.id_pedido = itens_pedido.id_pedido
JOIN livros ON itens_pedido.id_livro = livros.id_livro
GROUP BY pedidos.id_pedido
ORDER BY total_pedido DESC, pedidos.id_pedido
LIMIT 10
""")
cursor.fetchall()


[(40, 1703.96), (64, 1655.07), (193, 1649.48), (72, 1605.15), (236, 1590.03), (141, 1567.52), (227, 1559.6), (157, 1520.85), (42, 1495.7), (241, 1484.46)]

## Exercício 9 — Unidades vendidas por livro

Calcule a quantidade total de unidades vendidas de **cada livro**, incluindo os que nunca foram vendidos.

Mostre `id_livro`, `titulo` e `total_unidades`. Quem nunca foi vendido deve aparecer com **0**.

Ordene da maior quantidade para a menor; em caso de empate, pelo `id_livro`.

> `COALESCE(valor, 0)` usa 0 quando o resultado é NULL. Para contar unidades, some `quantidade`.


In [14]:
cursor.execute("""
SELECT livros.id_livro, livros.titulo,
       COALESCE(SUM(itens_pedido.quantidade), 0) AS total_unidades
FROM livros
LEFT JOIN itens_pedido ON livros.id_livro = itens_pedido.id_livro
GROUP BY livros.id_livro, livros.titulo
ORDER BY total_unidades DESC, livros.id_livro
""")
cursor.fetchall()


[(31, 'A sombra do jardim', 90), (25, 'O trem da madrugada', 85), (8, 'Memórias de um amanhã', 83), (35, 'O planeta das memórias', 81), (16, 'O relógio sem ponteiros', 74), (19, 'A biblioteca das marés', 74), (21, 'O canto da cidade', 72), (26, 'A estrada de vidro', 72), (27, 'Versos para o mar', 71), (17, 'Entre rios e montanhas', 68), (22, 'A ilha distante', 67), (28, 'O céu de outro planeta', 67), (20, 'O nome das flores', 66), (3, 'Cartas de inverno', 64), (30, 'O inverno das cartas', 64), (6, 'Caminhos de areia', 63), (5, 'O jardim secreto do tempo', 62), (33, 'Palavras ao entardecer', 62), (9, 'A ponte entre mundos', 60), (10, 'O silêncio da praça', 60), (12, 'Depois da chuva', 60), (18, 'O espelho da lua', 60), (24, 'Páginas de outubro', 60), (32, 'O último cometa', 60), (29, 'A janela amarela', 58), (7, 'O farol azul', 56), (34, 'A travessia do bosque', 56), (13, 'O viajante das nuvens', 54), (2, 'O mapa das estrelas', 52), (4, 'A última estação', 52), (23, 'O segredo do observ

## Exercício 10 — Faturamento por país do autor

Calcule o faturamento dos livros agrupado pelo **país de seus autores**, incluindo países sem venda com valor 0.

Mostre `pais` e `faturamento`, arredondado para 2 casas decimais. Ordene do maior faturamento para o menor, desempatando pelo país.

> Cada livro desta base tem um único autor. O país é o do autor, e não o do comprador.


In [15]:
cursor.execute("""
SELECT autores.pais,
       ROUND(COALESCE(SUM(itens_pedido.quantidade * livros.preco), 0), 2) AS faturamento
FROM autores
LEFT JOIN livros ON autores.id_autor = livros.id_autor
LEFT JOIN itens_pedido ON livros.id_livro = itens_pedido.id_livro
GROUP BY autores.pais
ORDER BY faturamento DESC, autores.pais
""")
cursor.fetchall()


[('Portugal', 45781.64), ('Brasil', 44043.93), ('Moçambique', 34421.5), ('Cabo Verde', 31173.62), ('Angola', 28735.6)]

## Exercício 11 — Pedidos e faturamento por autor

Para **todos os autores**, calcule o número de pedidos que contêm ao menos um de seus livros e o faturamento total de seus livros.

Mostre `id_autor`, `nome`, `total_pedidos` e `faturamento`, arredondado para 2 casas decimais. Autores sem venda devem aparecer com **0 pedidos e 0 de faturamento**.

Ordene do maior faturamento para o menor; em caso de empate, pelo `id_autor`.

> Use LEFT JOIN em toda a cadeia. `COUNT(DISTINCT pedidos.id_pedido)` conta um pedido uma única vez por autor, mesmo que ele tenha mais de um livro desse autor.


In [16]:
# Um mesmo pedido pode contar para autores diferentes. O faturamento soma somente os livros de cada autor.
cursor.execute("""
SELECT autores.id_autor, autores.nome,
       COUNT(DISTINCT pedidos.id_pedido) AS total_pedidos,
       ROUND(COALESCE(SUM(itens_pedido.quantidade * livros.preco), 0), 2) AS faturamento
FROM autores
LEFT JOIN livros ON autores.id_autor = livros.id_autor
LEFT JOIN itens_pedido ON livros.id_livro = itens_pedido.id_livro
LEFT JOIN pedidos ON itens_pedido.id_pedido = pedidos.id_pedido
GROUP BY autores.id_autor, autores.nome
ORDER BY faturamento DESC, autores.id_autor
""")
cursor.fetchall()


[(1, 'Ana Ribeiro', 66, 16289.51), (8, 'Henrique Dias', 57, 15272.46), (5, 'Elisa Rocha', 44, 14779.81), (14, 'Otavio Souza', 45, 13124.7), (12, 'Marcos Vieira', 47, 12884.72), (16, 'Rafael Gomes', 54, 12596.74), (15, 'Paula Fernandes', 40, 12072.72), (4, 'Diego Costa', 44, 11803.36), (17, 'Sabrina Melo', 46, 11785.92), (2, 'Bruno Santos', 51, 10800.08), (7, 'Gabriela Alves', 44, 10310.92), (6, 'Felipe Lima', 41, 10213.14), (9, 'Isabela Duarte', 55, 9493.44), (13, 'Natalia Lopes', 43, 7959.36), (3, 'Carla Mendes', 51, 5503.78), (11, 'Larissa Nunes', 42, 4944.54), (10, 'João Martins', 50, 4321.09), (18, 'Tiago Barros', 0, 0.0), (19, 'Vanessa Pinto', 0, 0.0), (20, 'Wesley Castro', 0, 0.0)]

## Exercício 12 — Faturamento por gênero

Calcule o faturamento de cada gênero literário, incluindo gêneros sem venda com valor **0**.

Mostre `genero` e `faturamento`, arredondado para 2 casas decimais. Ordene do maior para o menor faturamento; em caso de empate, pelo gênero.

Depois de rodar, responda: qual gênero gerou mais receita? Qual foi o menor? Explique por que **unidades vendidas** e **faturamento** medem coisas diferentes.


In [17]:
# Faturamento considera tanto o preço quanto a quantidade vendida.
cursor.execute("""
SELECT livros.genero,
       ROUND(COALESCE(SUM(itens_pedido.quantidade * livros.preco), 0), 2) AS faturamento
FROM livros
LEFT JOIN itens_pedido ON livros.id_livro = itens_pedido.id_livro
GROUP BY livros.genero
ORDER BY faturamento DESC, livros.genero
""")
cursor.fetchall()


[('Romance', 45978.25), ('Ficção científica', 40367.15), ('Poesia', 37019.28), ('Fantasia', 35065.34), ('Suspense', 25726.27)]

## Para pensar depois das consultas

- Em quais consultas INNER JOIN e LEFT JOIN produzem resultados diferentes? Por quê?
- Por que `COUNT(*)` depois de um JOIN com itens pode contar o mesmo pedido mais de uma vez?
- Por que o total de pedidos por autor pode somar mais que a quantidade de pedidos da livraria?
- No exercício 2, quando DEFAULT foi usado? Ele substituiria um NULL informado explicitamente?

Depois de terminar, feche a conexão na célula abaixo.


In [18]:
conexao.close()
